# Telecom Customer Churn Prediction
## Notebook 3: Modeling, Evaluation & Business Impact

**Author:** Vijayalakshmi Veeraiyan  
**Goal:** Train and compare machine learning models, optimise classification threshold and calculate business impact of the best model.

---

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report, accuracy_score,
    roc_auc_score, roc_curve, confusion_matrix
)
from sklearn.model_selection import RandomizedSearchCV
from sklearn.inspection import permutation_importance
from imblearn.over_sampling import SMOTE

print('Libraries loaded successfully!')

## 2. Load Processed Data

Run Notebook 2 first to generate these files.

In [ ]:
# Load processed data from Notebook 2
X_train = np.load('/content/X_train.npy')
X_test = np.load('/content/X_test.npy')
y_train = np.load('/content/y_train.npy')
y_test = np.load('/content/y_test.npy')
feature_names = joblib.load('/content/feature_names.pkl')

print(f'Data loaded successfully!')
print(f'Training set: {X_train.shape[0]} samples, {X_train.shape[1]} features')
print(f'Test set: {X_test.shape[0]} samples')
print(f'\nClass distribution in training set:')
print(pd.Series(y_train).value_counts(normalize=True).round(3))

## 3. Handle Class Imbalance with SMOTE

The dataset has ~73% non-churners and ~27% churners.
SMOTE (Synthetic Minority Oversampling Technique) creates synthetic samples for the minority class to balance training data.

In [ ]:
print('Handling class imbalance with SMOTE...')
print(f'Before SMOTE - Class distribution:')
print(pd.Series(y_train).value_counts(normalize=True).round(3))

smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

print(f'\nAfter SMOTE - Class distribution:')
print(pd.Series(y_train_resampled).value_counts(normalize=True).round(3))
print(f'\nTraining samples after SMOTE: {X_train_resampled.shape[0]}')

## 4. Model Comparison — Baseline Models

Compare three algorithms before hyperparameter tuning to identify the best base model.

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42)
}

results = {}

for name, model in models.items():
    print(f'Training {name}...')
    model.fit(X_train_resampled, y_train_resampled)

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    accuracy = accuracy_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_prob)

    results[name] = {
        'model': model,
        'accuracy': accuracy,
        'roc_auc': roc_auc
    }

    print(f'  Accuracy: {accuracy:.4f} | ROC AUC: {roc_auc:.4f}')
    print()

# Summary table
summary = pd.DataFrame({
    'Model': list(results.keys()),
    'Accuracy': [r['accuracy'] for r in results.values()],
    'ROC AUC': [r['roc_auc'] for r in results.values()]
})
print('\nModel Comparison Summary:')
print(summary.to_string(index=False))

In [ ]:
# Visualise model comparison
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

model_names = list(results.keys())
accuracies = [r['accuracy'] for r in results.values()]
roc_aucs = [r['roc_auc'] for r in results.values()]

axes[0].bar(model_names, accuracies, color=['steelblue', 'salmon', 'green'])
axes[0].set_title('Model Accuracy Comparison', fontsize=12)
axes[0].set_ylabel('Accuracy')
axes[0].set_ylim(0.75, 0.90)
axes[0].tick_params(axis='x', rotation=15)

axes[1].bar(model_names, roc_aucs, color=['steelblue', 'salmon', 'green'])
axes[1].set_title('Model ROC AUC Comparison', fontsize=12)
axes[1].set_ylabel('ROC AUC')
axes[1].set_ylim(0.80, 0.95)
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print('Gradient Boosting selected as best model — highest accuracy and ROC AUC')

## 5. Hyperparameter Tuning — Gradient Boosting

Using RandomizedSearchCV to find the best parameters for Gradient Boosting.
This uses 5-fold cross-validation and optimises for ROC AUC.

In [ ]:
print('Performing hyperparameter tuning...')
print('This may take a few minutes in Google Colab...')

param_grid = {
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [3, 4, 5],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'subsample': [0.8, 0.9, 1.0]
}

gb_clf = GradientBoostingClassifier(random_state=42)
random_search = RandomizedSearchCV(
    gb_clf,
    param_distributions=param_grid,
    n_iter=20,
    cv=5,
    scoring='roc_auc',
    random_state=42,
    n_jobs=-1,
    verbose=1
)

random_search.fit(X_train_resampled, y_train_resampled)

print(f'\nBest parameters: {random_search.best_params_}')
print(f'Best ROC AUC (cross-validation): {random_search.best_score_:.4f}')

best_model = random_search.best_estimator_

## 6. Train Final Model & Evaluate

In [ ]:
# Train final model
print('Training final model with best parameters...')
best_model.fit(X_train_resampled, y_train_resampled)

# Evaluate with default threshold (0.5)
y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:, 1]

print(f'\nDefault threshold (0.5) results:')
print(f'Accuracy: {accuracy_score(y_test, y_pred):.4f}')
print(f'ROC AUC: {roc_auc_score(y_test, y_prob):.4f}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred,
      target_names=['Not Churned', 'Churned']))

## 7. Feature Importance Analysis

In [ ]:
# Feature importance
importances = best_model.feature_importances_
feature_imp = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values('Importance', ascending=False).head(15)

plt.figure(figsize=(12, 8))
sns.barplot(x='Importance', y='Feature', data=feature_imp, color='steelblue')
plt.title('Top 15 Feature Importances — Gradient Boosting', fontsize=13)
plt.xlabel('Importance Score')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print('Top 10 features:')
print(feature_imp.head(10).to_string(index=False))

## 8. ROC Curve & Threshold Optimisation

The default threshold of 0.5 maximises accuracy but may miss many actual churners.
We use Youden's J statistic to find the threshold that best balances sensitivity and specificity.

In [ ]:
# Calculate ROC curve
fpr, tpr, thresholds = roc_curve(y_test, y_prob)
roc_auc = roc_auc_score(y_test, y_prob)

# Find optimal threshold using Youden's J statistic
optimal_idx = np.argmax(tpr - fpr)
optimal_threshold = thresholds[optimal_idx]

print(f'Optimal classification threshold: {optimal_threshold:.4f}')
print(f'(Default threshold was 0.5)')

# Plot ROC curve
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='steelblue', linewidth=2,
         label=f'ROC curve (AUC = {roc_auc:.4f})')
plt.scatter(fpr[optimal_idx], tpr[optimal_idx],
            marker='o', color='red', s=100,
            label=f'Optimal threshold: {optimal_threshold:.4f}')
plt.plot([0, 1], [0, 1], 'k--', linewidth=1)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve with Optimal Threshold', fontsize=13)
plt.legend()
plt.tight_layout()
plt.savefig('roc_curve.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Evaluate at optimal threshold
y_pred_optimal = (y_prob >= optimal_threshold).astype(int)

print(f'Results at optimal threshold ({optimal_threshold:.4f}):')
print(f'Accuracy: {accuracy_score(y_test, y_pred_optimal):.4f}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred_optimal,
      target_names=['Not Churned', 'Churned']))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred_optimal)
tn, fp, fn, tp = cm.ravel()

plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Not Churned', 'Churned'],
            yticklabels=['Not Churned', 'Churned'],
            annot_kws={'size': 14})
plt.xlabel('Predicted', fontsize=12)
plt.ylabel('Actual', fontsize=12)
plt.title(f'Confusion Matrix at Optimal Threshold ({optimal_threshold:.4f})', fontsize=12)
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nConfusion Matrix Results:')
print(f'  True Positives (correctly identified churners): {tp}')
print(f'  False Positives (unnecessary retention offers): {fp}')
print(f'  False Negatives (missed churners): {fn}')
print(f'  True Negatives (correctly identified non-churners): {tn}')

## 9. Business Impact Analysis

Translating model performance into measurable business value.

### Assumptions (based on typical telecom industry benchmarks)

| Parameter | Value | Rationale |
|---|---|---|
| Monthly revenue per customer | $70 | Based on average Monthly Charge in dataset |
| Average customer lifetime | 24 months | Typical telecom contract duration |
| Customer Lifetime Value (CLV) | $1,680 | $70 × 24 months |
| Retention offer cost | $50 | Typical discount/incentive offer |

> **Note:** These are assumed benchmarks. Actual business impact depends on real company-specific values.

In [ ]:
# Business assumptions
monthly_revenue = 70      # $ per customer per month
avg_lifetime = 24         # months
retention_cost = 50       # $ per retention offer

# Customer Lifetime Value
clv = monthly_revenue * avg_lifetime

# Business impact calculation
value_retained = tp * (clv - retention_cost)   # correctly identified churners saved
cost_unnecessary = fp * retention_cost           # offers sent to customers who wouldn't churn
cost_missed = fn * clv                           # churners we missed — lost full CLV

total_retention_spend = (tp + fp) * retention_cost
net_benefit = value_retained - cost_unnecessary
roi = (value_retained - total_retention_spend) / total_retention_spend * 100

print('=' * 50)
print('BUSINESS IMPACT ANALYSIS')
print('=' * 50)
print(f'\nAssumptions:')
print(f'  Monthly revenue per customer: ${monthly_revenue}')
print(f'  Average customer lifetime: {avg_lifetime} months')
print(f'  Customer Lifetime Value (CLV): ${clv:,}')
print(f'  Retention offer cost: ${retention_cost}')
print(f'\nModel Performance at Threshold {optimal_threshold:.4f}:')
print(f'  True Positives (churners identified): {tp}')
print(f'  False Positives (unnecessary offers): {fp}')
print(f'  False Negatives (missed churners): {fn}')
print(f'\nFinancial Impact:')
print(f'  Value from retained customers: ${value_retained:,.2f}')
print(f'  Cost of unnecessary offers: ${cost_unnecessary:,.2f}')
print(f'  Cost of missed churners: ${cost_missed:,.2f}')
print(f'\n  Net Benefit: ${net_benefit:,.2f}')
print(f'  ROI: {roi:.2f}%')
print('=' * 50)

In [ ]:
# Visualise business impact
categories = ['Value Retained', 'Unnecessary Costs', 'Missed Churner Costs']
values = [value_retained, -cost_unnecessary, -cost_missed]
colors = ['green', 'salmon', 'orange']

plt.figure(figsize=(10, 6))
bars = plt.bar(categories, values, color=colors)
plt.axhline(y=0, color='black', linewidth=0.8)
plt.title('Business Impact Breakdown', fontsize=13)
plt.ylabel('Financial Impact ($)')

# Add value labels
for bar, val in zip(bars, values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2000,
             f'${abs(val):,.0f}', ha='center', va='bottom', fontsize=11)

plt.tight_layout()
plt.savefig('business_impact.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Save Final Model

In [ ]:
# Save model and threshold
joblib.dump(best_model, '/content/best_churn_model.pkl')
joblib.dump(optimal_threshold, '/content/optimal_threshold.pkl')

print('Model saved successfully!')
print('Files saved:')
print('  best_churn_model.pkl')
print('  optimal_threshold.pkl')
print('\nTo use this model on new data:')
print('  model = joblib.load("best_churn_model.pkl")')
print('  threshold = joblib.load("optimal_threshold.pkl")')
print('  y_pred = (model.predict_proba(X_new)[:, 1] >= threshold).astype(int)')

## 11. Project Summary

### Model Performance

| Metric | Value |
|---|---|
| Algorithm | Gradient Boosting (tuned) |
| Accuracy at optimal threshold | 82.61% |
| ROC AUC | ~0.96 |
| Optimal threshold | 0.2735 |
| Precision (non-churners) | 93% |
| Recall (churners) | 82% |

### Key Findings

1. **Month-to-Month contracts** are the strongest churn predictor (importance: 0.384)
2. **Number of referrals** strongly signals customer loyalty (importance: 0.108)
3. **Monthly Spend Ratio** — high spend relative to tenure increases churn risk (importance: 0.073)
4. **SMOTE** significantly improved recall for the minority (churned) class
5. **Threshold optimisation** from 0.5 to 0.273 improved business value by catching more actual churners

### Business Impact (assumed benchmarks)

| Metric | Value |
|---|---|
| Value from retained customers | $498,780 |
| Cost of retention offers | $8,900 |
| **Net benefit** | **$377,320** |
| **ROI** | **1,961%** |

> These figures use assumed telecom industry benchmarks ($70/month, 24-month lifetime, $50 retention cost). Actual impact depends on real company-specific values.